# №3 Object detection, object tracking

pip install ultralytics

## Object detection

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"


In [2]:
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

sns.set_theme()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/dan/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### PHOTO

#### YOLO

In [ ]:
# read image
img_dog_bike =

In [ ]:
# initialize YOLO model
model_yolo =

In [ ]:
# apply model
result_yolo =

In [ ]:
res_img_yolo =

In [ ]:
annotated_res_img_yolo =

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(annotated_res_img_yolo)
plt.axis("off")
plt.show()

#### Faster R-CNN

In [ ]:
import torch
import torchvision

from torchvision.transforms import functional

In [ ]:
# initialize RCNN model
model_rcnn =

In [ ]:
# evaluation

In [ ]:
# run on CPU
device =


In [ ]:
# read image
img_dog_bike =

In [ ]:
# convert image to tensor
img_dog_bike_tensor =
img_dog_bike_tensor

In [ ]:
with torch.no_grad():
    predictions =

predictions

In [ ]:
confidence_threshold =

pred =

# create mask
mask =

In [ ]:
# filter boxes, scores, labels



In [ ]:
labels_text = [f"ID {label.item()}: {score:.2f}" for label, score in zip(filtered_labels, filtered_scores)]

In [ ]:
# Готуємо uint8 тензор [C, H, W] для малювання рамок
# (перетворюємо NumPy масив у тензор та міняємо осі HWC -> CHW)

img_tensor_uint8 =

In [ ]:
result_tensor = torchvision.utils.draw_bounding_boxes(
    image=,
    boxes=,
    labels=,
    colors="green",
    width=2
)

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(result_tensor.permute(1, 2, 0).cpu().numpy())
plt.axis('off')
plt.show()

Виділення bboxes з назвами класів

In [ ]:
weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
weights

In [ ]:
COCO_CLASSES =
COCO_CLASSES

In [ ]:
labels_text = []
for label_id, score in zip(filtered_labels, filtered_scores):
    class_name = COCO_CLASSES[label_id]

    text = f"{class_name}: {score.item():.2f}"
    labels_text.append(text)

In [ ]:
result_tensor = torchvision.utils.draw_bounding_boxes(
    image=img_tensor_uint8,    # uint8 [C, H, W]
    boxes=filtered_boxes,
    labels=labels_text,
    colors="green",
    width=2
)

plt.figure(figsize=(10, 8))
plt.imshow(result_tensor.permute(1, 2, 0).cpu().numpy())
plt.axis('off')
plt.show()

### Video

#### YOLO

In [ ]:
# read video
video_traffic =

In [ ]:
fps =
width  =
height =

fps, width, height

In [ ]:
# Configure the output video recording
fourcc =

out =


In [ ]:
# create new video with annotations

while video_traffic.isOpened():


## Object tracking

### YOLO

In [ ]:
# define model
model_yolo =

In [ ]:
# read video
video_store =

In [ ]:
fps = int(video_store.get(cv2.CAP_PROP_FPS))
width = int(video_store.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video_store.get(cv2.CAP_PROP_FRAME_HEIGHT))

fps, height, width

In [ ]:
output_video_path = 'output_tracked_store.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

In [ ]:
ret, frame = video_store.read()

In [ ]:
while video_store.isOpened():
    ret, frame = video_store.read()
    if not ret:
        break

    # track object
    results =

    annotated_frame = results[0].plot()

    out.write(annotated_frame)

video_store.release()
out.release()

print(f"Трекінг завершено! Відео збережено у файл: {output_video_path}")

### Оптичний потік (Optical Flow) — алгоритм Lucas-Kanade

Алгоритм Лукаса-Канаде  визначає, куди перемістилися пікселі об'єкта від попереднього кадру до поточного.
<br>
В основі алгоритму лежать 3 прості припущення:
1. Колір та яскравість пікселя об'єкта залишаються однаковими від кадру до кадру.
2. Об'єкт у відео рухається плавно (не робить миттєвих "телепортацій" на велику відстань).
3. Сусідній рух: Усі сусідні пікселі у невеличкій області (віконці) рухаються разом в одному напрямку і з однаковою швидкістю.
<br>
Замість стеження за кожним пікселем кадру, ми спочатку знаходимо лише найбільш контрастні точки (кути, текстурні плями), а алгоритм Лукаса-Канаде математично обчислює вектор їхнього зсуву на кожному новому кадрі.

#### 1. Рівняння оптичного потоку ($I(x, y, t) = I(x+dx, y+dy, t+dt)$)

Вважаючи, що яскравість пікселя не змінюється під час руху, отримуємо базове рівняння:

$$I_x u + I_y v + I_t = 0$$

* $I_x, I_y$ — просторові градієнти (зміна яскравості по $x$ та $y$).
* $I_t$ — часовий градієнт (зміна яскравості в часі).
* $u, v$ — вектор швидкості руху точки (невідомі).

---

#### 2. Проблема апертури (Aperture Problem)

Одне рівняння має **два невідомих** ($u$ та $v$), тому локально визначити точний напрямок руху неможливо.

---

#### 3. Метод Лукаса-Канаде

Припускає, що всі пікселі в невеликому вікні ($3\times3$ або $5\times5$) рухаються з **однаковою швидкістю**. Це дає систему з $n$ рівнянь для двох невідомих:

$$\begin{bmatrix} I_{x1} & I_{y1} \\ \vdots & \vdots \\ I_{xn} & I_{yn} \end{bmatrix} \begin{bmatrix} u \\ v \end{bmatrix} = -\begin{bmatrix} I_{t1} \\ \vdots \\ I_{tn} \end{bmatrix} \quad \implies \quad A \cdot V = b$$

---

#### 4. Розв'язок (метод найменших квадратів)

Система вирішується через псевдообернену матрицю:

$$V = (A^T A)^{-1} A^T b, \quad \text{де } A^T A = \begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix}$$

**Головна умова:** матриця $A^T A$ повинна бути оборотною. Це означає, що у вікні має бути достатня текстура в обох напрямках (кути або чіткі контури). Метод краще працює для **кутів**, ніж для рівних ділянок чи прямих ліній.

In [ ]:
video_path = 'data/4k-road-traffic-5sec.mp4'  # Вкажи свій шлях
cap = cv2.VideoCapture(video_path)

In [ ]:
# ret, old_frame = cap.read()
# if not ret:
#     print("Не вдалося відкрити відео")
#     exit()
#
# old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
#
# bbox = cv2.selectROI("Optical Flow Tracker", old_frame, False)
# cv2.destroyWindow("Optical Flow Tracker")
#
# print(f"Your BBOX:  {bbox}")
#
# x, y, w, h = map(int, bbox)
#
# # Вирізаємо маску для об'єкта та шукаємо ключові точки всередині рамки
# mask = np.zeros_like(old_gray)
# mask[y:y+h, x:x+w] = 255
#
# # Шукаємо кути
# p0 = cv2.goodFeaturesToTrack(
#     old_gray,
#     mask=mask,
#     maxCorners=50,      # Максимум точок для відстеження
#     qualityLevel=0.3,   # Поріг якості точок
#     minDistance=7       # Мінімальна відстань між точками
# )
#
# # Параметри для оптичного потоку Lucas-Kanade
# lk_params = dict(
#     winSize=(15, 15),
#     maxLevel=2,
#     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
# )
#
# print("Розпочинаємо трекінг...")
#
#
# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break
#
#     frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
#
#     if p0 is not None and len(p0) > 0:
#         # Обчислюємо оптичний потік (куди перемістилися точки)
#         p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
#
#         # Відбираємо лише успішно знайдені точки
#         if p1 is not None:
#             good_new = p1[st == 1]
#             good_old = p0[st == 1]
#
#             # Малюємо точки та їхній рух
#             for new in good_new:
#                 a, b = new.ravel()
#                 cv2.circle(frame, (int(a), int(b)), 4, (0, 255, 0), -1)
#
#             # Оновлюємо Bounding Box навколо точок
#             if len(good_new) > 0:
#                 x_min, y_min = np.min(good_new, axis=0)
#                 x_max, y_max = np.max(good_new, axis=0)
#                 cv2.rectangle(frame, (int(x_min)-5, int(y_min)-5), (int(x_max)+5, int(y_max)+5), (0, 255, 0), 2)
#
#             # Готуємо точки для наступного кадру
#             p0 = good_new.reshape(-1, 1, 2)
#     else:
#         cv2.putText(frame, "LOST!", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
#
#     # Оновлюємо попередній кадр
#     old_gray = frame_gray.copy()
#
#     cv2.imshow("Optical Flow Tracker", frame)
#     if cv2.waitKey(30) & 0xFF == ord('q'):
#         break
#
# cap.release()
# cv2.destroyAllWindows()